# LSI và tóm tắt dựa trên không gian ngữ nghĩa

## Thư viện

In [32]:
import re
import unicodedata
from collections import Counter

import numpy as np
import pandas as pd
from datasets import load_dataset

# Phần 1: Mô phỏng

## 1.1. Văn bản đầu vào

In [33]:
demo_text = """Máy tính xử lý ngôn ngữ.
Ngôn ngữ giúp máy tính hiểu văn bản.
LSI giúp tóm tắt văn bản.
Tóm tắt chọn câu trong văn bản.
Đội bóng thi đấu bóng đá.
Bóng đá cần đội bóng mạnh."""

print(demo_text)

Máy tính xử lý ngôn ngữ.
Ngôn ngữ giúp máy tính hiểu văn bản.
LSI giúp tóm tắt văn bản.
Tóm tắt chọn câu trong văn bản.
Đội bóng thi đấu bóng đá.
Bóng đá cần đội bóng mạnh.


## 1.2. Tách câu và tiền xử lý

In [34]:
STOP_WORDS = set("""
a an the and or but is are was were be been being to of in on at for from
with by as that this these those it its he she they we you i his her their
our not do does did have has had will would can could should may might
và là của có được một những các cho với đã đang sẽ này đó ở để thì mà
""".split())


def split_sentences(text):
    text = unicodedata.normalize("NFC", text)
    parts = re.split(r'(?<=[.!?])\s+|(?<=[.!?]["”’])\s+|\n+', text)
    return list(filter(None, map(str.strip, parts)))


def tokenize(text):
    text = unicodedata.normalize("NFC", text).lower()
    return re.findall(r"\b\w+\b", text)


def preprocess(sentence):
    return list(filter(lambda word: word not in STOP_WORDS, tokenize(sentence)))

In [35]:
demo_sentences = split_sentences(demo_text)
demo_tokens = list(map(preprocess, demo_sentences))
sentence_names = [f"S{index + 1}" for index in range(len(demo_sentences))]

In [36]:
pd.DataFrame({"Câu": demo_sentences, "Token sau tiền xử lý": demo_tokens}, index=sentence_names)

,Câu,Token sau tiền xử lý
S1,Máy tính xử lý ngôn ngữ.,"[máy, tính, xử, lý, ngôn, ngữ]"
S2,Ngôn ngữ giúp máy tính hiểu văn bản.,"[ngôn, ngữ, giúp, máy, tính, hiểu, văn, bản]"
S3,LSI giúp tóm tắt văn bản.,"[lsi, giúp, tóm, tắt, văn, bản]"
S4,Tóm tắt chọn câu trong văn bản.,"[tóm, tắt, chọn, câu, trong, văn, bản]"
S5,Đội bóng thi đấu bóng đá.,"[đội, bóng, thi, đấu, bóng, đá]"
S6,Bóng đá cần đội bóng mạnh.,"[bóng, đá, cần, đội, bóng, mạnh]"


## 1.3. Tự cài đặt TF-IDF

Với $N$ câu và từ $t$:

$$TF(t,s_j)=\text{số lần từ }t\text{ xuất hiện trong câu }s_j$$
$$DF(t)=\text{số câu chứa từ }t$$
$$IDF(t)=\ln\frac{N+1}{DF(t)+1}+1$$
$$W_{t,j}=TF(t,s_j)\,IDF(t)$$
$$A_{t,j}=\frac{W_{t,j}}{\sqrt{\sum_t W_{t,j}^{2}}}$$

In [37]:
def build_tfidf(sentences):
    tokenized_sentences = list(map(preprocess, sentences))
    vocabulary = sorted(set(word for tokens in tokenized_sentences for word in tokens))

    counts = list(map(Counter, tokenized_sentences))
    tf = np.array([[count[word] for count in counts] for word in vocabulary], dtype=float)
    df = np.count_nonzero(tf, axis=1)
    idf = np.log((len(sentences) + 1) / (df + 1)) + 1
    weights = tf * idf[:, np.newaxis]
    column_norms = np.linalg.norm(weights, axis=0)
    matrix = weights / np.maximum(column_norms, np.finfo(float).eps)

    return {"vocabulary": vocabulary, "tf": tf, "df": df, "idf": idf,
            "weights": weights, "matrix": matrix}

In [38]:
demo_tfidf = build_tfidf(demo_sentences)

### TF

In [39]:
pd.DataFrame(demo_tfidf["tf"], index=demo_tfidf["vocabulary"], columns=sentence_names)

,S1,S2,S3,S4,S5,S6
bóng,0.0,0.0,0.0,0.0,2.0,2.0
bản,0.0,1.0,1.0,1.0,0.0,0.0
chọn,0.0,0.0,0.0,1.0,0.0,0.0
câu,0.0,0.0,0.0,1.0,0.0,0.0
cần,0.0,0.0,0.0,0.0,0.0,1.0
giúp,0.0,1.0,1.0,0.0,0.0,0.0
hiểu,0.0,1.0,0.0,0.0,0.0,0.0
lsi,0.0,0.0,1.0,0.0,0.0,0.0
lý,1.0,0.0,0.0,0.0,0.0,0.0
máy,1.0,1.0,0.0,0.0,0.0,0.0


### DF và IDF

In [40]:
pd.DataFrame({"DF": demo_tfidf["df"], "IDF": demo_tfidf["idf"]}, index=demo_tfidf["vocabulary"])

,DF,IDF
bóng,2,1.847298
bản,3,1.559616
chọn,1,2.252763
câu,1,2.252763
cần,1,2.252763
giúp,2,1.847298
hiểu,1,2.252763
lsi,1,2.252763
lý,1,2.252763
máy,2,1.847298


### TF × IDF

In [41]:
pd.DataFrame(demo_tfidf["weights"], index=demo_tfidf["vocabulary"], columns=sentence_names)

,S1,S2,S3,S4,S5,S6
bóng,0.000000,0.000000,0.000000,0.000000,3.694596,3.694596
bản,0.000000,1.559616,1.559616,1.559616,0.000000,0.000000
chọn,0.000000,0.000000,0.000000,2.252763,0.000000,0.000000
câu,0.000000,0.000000,0.000000,2.252763,0.000000,0.000000
cần,0.000000,0.000000,0.000000,0.000000,0.000000,2.252763
giúp,0.000000,1.847298,1.847298,0.000000,0.000000,0.000000
hiểu,0.000000,2.252763,0.000000,0.000000,0.000000,0.000000
lsi,0.000000,0.000000,2.252763,0.000000,0.000000,0.000000
lý,2.252763,0.000000,0.000000,0.000000,0.000000,0.000000
máy,1.847298,1.847298,0.000000,0.000000,0.000000,0.000000


### Ma trận A: chuẩn hóa L2

In [42]:
pd.DataFrame(demo_tfidf["matrix"], index=demo_tfidf["vocabulary"], columns=sentence_names)

,S1,S2,S3,S4,S5,S6
bóng,0.000000,0.000000,0.000000,0.000000,0.667620,0.667620
bản,0.000000,0.300135,0.347205,0.300624,0.000000,0.000000
chọn,0.000000,0.000000,0.000000,0.434231,0.000000,0.000000
câu,0.000000,0.000000,0.000000,0.434231,0.000000,0.000000
cần,0.000000,0.000000,0.000000,0.000000,0.000000,0.407078
giúp,0.000000,0.355498,0.411250,0.000000,0.000000,0.000000
hiểu,0.000000,0.433526,0.000000,0.000000,0.000000,0.000000
lsi,0.000000,0.000000,0.501515,0.000000,0.000000,0.000000
lý,0.461772,0.000000,0.000000,0.000000,0.000000,0.000000
máy,0.378660,0.355498,0.000000,0.000000,0.000000,0.000000


## 1.4. SVD và không gian ngữ nghĩa

$$A=U\Sigma V^\top \approx U_k\Sigma_kV_k^\top$$

Giữ $k$ giá trị kỳ dị lớn nhất. Tọa độ ngữ nghĩa của các câu là:

$$Z=\Sigma_kV_k^\top$$

In [43]:
demo_k = 2
demo_u, demo_singular_values, demo_vt = np.linalg.svd(demo_tfidf["matrix"], full_matrices=False)
demo_k = min(demo_k, len(demo_singular_values))
dimension_names = [f"Chiều {index + 1}" for index in range(len(demo_singular_values))]

### U

In [44]:
pd.DataFrame(demo_u, index=demo_tfidf["vocabulary"], columns=dimension_names)

,Chiều 1,Chiều 2,Chiều 3,Chiều 4,Chiều 5,Chiều 6
bóng,-6.478705e-17,7.309228e-01,-4.924865e-17,6.592563e-17,5.100963e-17,1.291558e-16
bản,-3.724003e-01,-5.743496e-17,-1.709097e-01,-9.007848e-02,1.388312e-01,-8.539483e-18
chọn,-1.481154e-01,-7.258406e-17,-1.941456e-01,4.102069e-01,1.551147e-01,-1.037760e-17
câu,-1.481154e-01,-8.218918e-17,-1.941456e-01,4.102069e-01,1.551147e-01,-1.166350e-17
cần,-1.897296e-17,2.228384e-01,-2.197440e-17,1.638356e-17,-5.880470e-18,-5.000000e-01
giúp,-3.196353e-01,4.615466e-18,-4.323306e-02,-4.430696e-01,3.724331e-02,3.453992e-18
hiểu,-1.897584e-01,-1.153394e-17,1.369496e-01,-1.883090e-01,4.810706e-01,-9.750553e-19
lsi,-2.000340e-01,1.784181e-17,-1.896719e-01,-3.520104e-01,-4.356527e-01,3.662855e-18
lý,-1.330243e-01,5.737523e-17,2.646208e-01,2.370089e-01,-4.185034e-01,2.668668e-17
máy,-2.646865e-01,4.340261e-17,3.292935e-01,3.993459e-02,5.130598e-02,1.913236e-17


### Giá trị kỳ dị

In [45]:
pd.DataFrame({"sigma": demo_singular_values}, index=dimension_names)

,sigma
Chiều 1,1.348385
Chiều 2,1.291733
Chiều 3,1.138781
Chiều 4,0.737777
Chiều 5,0.583715
Chiều 6,0.575696


### Vᵀ

In [46]:
pd.DataFrame(demo_vt, index=dimension_names, columns=sentence_names)

,S1,S2,S3,S4,S5,S6
Chiều 1,-0.388434,-0.590201,-5.378155e-01,-4.599313e-01,-6.737607e-17,-6.284505e-17
Chiều 2,-0.000000,-0.000000,-6.078784e-17,-1.291228e-16,7.071068e-01,7.071068e-01
Chiều 3,0.652584,0.359737,-4.306840e-01,-5.091508e-01,-6.852694e-17,-6.147225e-17
Chiều 4,0.378671,-0.320465,-5.178409e-01,6.969585e-01,5.305928e-17,2.969310e-17
Chiều 5,-0.529020,0.647730,-5.070568e-01,2.085126e-01,2.917824e-18,-8.432079e-18
Chiều 6,-0.000000,-0.000000,8.753850e-18,-1.720229e-17,7.071068e-01,-7.071068e-01


In [47]:
demo_coordinates = demo_singular_values[:demo_k, np.newaxis] * demo_vt[:demo_k]
demo_approximation = demo_u[:, :demo_k] @ demo_coordinates

### Z: tọa độ ngữ nghĩa của các câu

In [48]:
pd.DataFrame(demo_coordinates, index=dimension_names[:demo_k], columns=sentence_names)

,S1,S2,S3,S4,S5,S6
Chiều 1,-0.523758,-0.795817,-7.251822e-01,-6.201643e-01,-9.084886e-17,-8.473930e-17
Chiều 2,-0.000000,-0.000000,-7.852167e-17,-1.667921e-16,9.133933e-01,9.133933e-01


### Aₖ: ma trận xấp xỉ

In [49]:
pd.DataFrame(demo_approximation, index=demo_tfidf["vocabulary"], columns=sentence_names)

,S1,S2,S3,S4,S5,S6
bóng,3.393276e-17,5.155867e-17,-1.041086e-17,-8.173356e-17,6.676200e-01,6.676200e-01
bản,1.950478e-01,2.963626e-01,2.700581e-01,2.309494e-01,-1.862857e-17,-2.090377e-17
chọn,7.757670e-02,1.178728e-01,1.074107e-01,9.185590e-02,-5.284168e-17,-5.374660e-17
câu,7.757670e-02,1.178728e-01,1.074107e-01,9.185590e-02,-6.161493e-17,-6.251985e-17
cần,9.937248e-18,1.509901e-17,-3.738790e-18,-2.540134e-17,2.035391e-01,2.035391e-01
giúp,1.674117e-01,2.543714e-01,2.317939e-01,1.982264e-01,3.325424e-17,3.130141e-17
hiểu,9.938756e-02,1.510131e-01,1.376094e-01,1.176814e-01,6.704310e-18,5.544970e-18
lsi,1.047695e-01,1.591905e-01,1.450611e-01,1.240539e-01,3.446945e-17,3.324733e-17
lý,6.967262e-02,1.058631e-01,9.646689e-02,8.249695e-02,6.449126e-17,6.367854e-17
máy,1.386318e-01,2.106421e-01,1.919459e-01,1.641491e-01,6.369012e-17,6.207300e-17


## 1.5. Cosine similarity và chọn câu

Vector đại diện văn bản là trung bình các vector câu:

$$c=\frac{1}{N}\sum_{j=1}^{N}z_j$$

$$score(s_j)=\frac{z_j^\top c}{\|z_j\|_2\|c\|_2}$$

Chọn câu có cosine cao nhất và giữ thứ tự xuất hiện trong văn bản.
Với `k=1`, các vector cùng hướng có cosine bằng 1 nên khó phân biệt câu.

In [50]:
def semantic_scores(singular_values, vt, k):
    k = min(k, len(singular_values))
    coordinates = singular_values[:k, np.newaxis] * vt[:k]
    centroid = coordinates.mean(axis=1)
    denominator = np.linalg.norm(coordinates, axis=0) * np.linalg.norm(centroid)
    return (centroid @ coordinates) / np.maximum(denominator, np.finfo(float).eps)


def select_sentences(sentences, scores, sentence_count):
    ranking = np.argsort(-np.round(scores, 12), kind="stable")
    selected_indices = np.sort(ranking[:sentence_count])
    return selected_indices, " ".join(sentences[index] for index in selected_indices)

### Vector đại diện văn bản

In [51]:
pd.DataFrame({"Trung bình": demo_coordinates.mean(axis=1)}, index=dimension_names[:demo_k])

,Trung bình
Chiều 1,-0.444154
Chiều 2,0.304464


In [52]:
demo_sentence_count = 2
demo_scores = semantic_scores(demo_singular_values, demo_vt, demo_k)
demo_selected, demo_summary = select_sentences(demo_sentences, demo_scores, demo_sentence_count)
demo_ranking = pd.DataFrame({"Câu": demo_sentences, "Cosine": demo_scores,
                             "Được chọn": np.isin(np.arange(len(demo_sentences)), demo_selected)},
                            index=sentence_names)

In [53]:
demo_ranking.sort_values("Cosine", ascending=False, kind="stable")

,Câu,Cosine,Được chọn
S3,LSI giúp tóm tắt văn bản.,0.824814,False
S1,Máy tính xử lý ngôn ngữ.,0.824814,True
S2,Ngôn ngữ giúp máy tính hiểu văn bản.,0.824814,True
S4,Tóm tắt chọn câu trong văn bản.,0.824814,False
S5,Đội bóng thi đấu bóng đá.,0.565404,False
S6,Bóng đá cần đội bóng mạnh.,0.565404,False


### Bản tóm tắt

In [54]:
print(demo_summary)

Máy tính xử lý ngôn ngữ. Ngôn ngữ giúp máy tính hiểu văn bản.


# Phần 2: Tóm tắt văn bản trích xuất

In [55]:
def summarize_lsi(text, k=2, sentence_count=3):
    sentences = split_sentences(text)
    tfidf = build_tfidf(sentences)
    u, singular_values, vt = np.linalg.svd(tfidf["matrix"], full_matrices=False)
    scores = semantic_scores(singular_values, vt, k)
    selected_indices, summary = select_sentences(sentences, scores, sentence_count)
    details = pd.DataFrame({
        "Câu": sentences,
        "Cosine": scores,
        "Được chọn": np.isin(np.arange(len(sentences)), selected_indices),
    }, index=[f"S{index + 1}" for index in range(len(sentences))])
    return summary, details

## Văn bản đầu vào

Thay `input_text`, `k` và `sentence_count` để tóm tắt văn bản khác.

In [56]:
input_text = demo_text
k = 2
sentence_count = 3

summary, sentence_details = summarize_lsi(input_text, k, sentence_count)

In [57]:
sentence_details

,Câu,Cosine,Được chọn
S1,Máy tính xử lý ngôn ngữ.,0.824814,True
S2,Ngôn ngữ giúp máy tính hiểu văn bản.,0.824814,True
S3,LSI giúp tóm tắt văn bản.,0.824814,True
S4,Tóm tắt chọn câu trong văn bản.,0.824814,False
S5,Đội bóng thi đấu bóng đá.,0.565404,False
S6,Bóng đá cần đội bóng mạnh.,0.565404,False


## Bản tóm tắt

In [58]:
print(summary)

Máy tính xử lý ngôn ngữ. Ngôn ngữ giúp máy tính hiểu văn bản. LSI giúp tóm tắt văn bản.


# Phần 3: Thử nghiệm và đánh giá

## 3.1. CNN/DailyMail

Nguồn: [abisee/cnn_dailymail](https://huggingface.co/datasets/abisee/cnn_dailymail).
Dùng 50 mẫu trong tập test.
`article`: bài báo; `highlights`: tóm tắt tham chiếu; `id`: mã bài báo.


In [59]:
records = load_dataset("abisee/cnn_dailymail", "3.0.0", split="test[:50]")
print("Số bài báo:", len(records))

Số bài báo: 50


In [60]:
pd.DataFrame(records[:3])[["id", "article", "highlights"]]

,id,article,highlights
0,f001ec5c4704938247d27a44948eebb37ae98d01,(CNN)The Palestinian Authority officially beca...,Membership gives the ICC jurisdiction over all...
1,230c522854991d053fe98a718b1defa077a8efef,(CNN)Never mind cats having nine lives. A stra...,"Theia, a bully breed mix, was apparently hit b..."
2,4495ba8f3a340d97a9df1476f8a35502bcce1f69,"(CNN)If you've been following the news lately,...",Mohammad Javad Zarif has spent more time with ...


## 3.2. Tự cài đặt ROUGE-1, ROUGE-2 và ROUGE-L

ROUGE-1/2 đếm unigram/bigram chung, có tính số lần xuất hiện.
ROUGE-L dùng độ dài chuỗi con chung dài nhất (LCS) của chuỗi token.

Với $M$ là số đơn vị chung, $C$ là số đơn vị trong đầu ra và $R$ trong tham chiếu:

$$P=M/C,\quad Recall=M/R,\quad F1=2M/(C+R)$$

Tính F1 trên token chữ thường, không stemming. ROUGE-L dùng LCS trên toàn chuỗi token.

In [61]:
def ngram_counts(tokens, n):
    return Counter(tuple(tokens[index:index + n]) for index in range(len(tokens) - n + 1))


def lcs_length(first, second):
    previous_row = [0] * (len(second) + 1)
    for first_token in first:
        current_row = [0]
        for column, second_token in enumerate(second, 1):
            current_row.append(max(
                previous_row[column],
                current_row[column - 1],
                previous_row[column - 1] + int(first_token == second_token),
            ))
        previous_row = current_row
    return previous_row[-1]


def rouge_scores(reference, prediction):
    reference_tokens = tokenize(reference)
    prediction_tokens = tokenize(prediction)
    scores = {}
    for n in (1, 2):
        reference_ngrams = ngram_counts(reference_tokens, n)
        prediction_ngrams = ngram_counts(prediction_tokens, n)
        overlap = sum((reference_ngrams & prediction_ngrams).values())
        total = sum(reference_ngrams.values()) + sum(prediction_ngrams.values())
        scores[f"ROUGE-{n}"] = 2 * overlap / max(total, 1)

    common_length = lcs_length(reference_tokens, prediction_tokens)
    total_length = len(reference_tokens) + len(prediction_tokens)
    scores["ROUGE-L"] = 2 * common_length / max(total_length, 1)
    return scores

## 3.3. Thí nghiệm

Mỗi phương pháp lấy tối đa 3 câu:

- Lead-3: ba câu đầu.
- TF-IDF: tổng trọng số TF-IDF.
- LSI: điểm trong không gian ngữ nghĩa với `k = 1, 2, 3`.

In [62]:
experiment_sentence_count = 3
experiment_k_values = [1, 2, 3]
evaluation_rows = []
summary_rows = []

for record in records:
    sentences = split_sentences(record["article"])
    tfidf = build_tfidf(sentences)
    u, singular_values, vt = np.linalg.svd(tfidf["matrix"], full_matrices=False)

    tfidf_indices, tfidf_summary = select_sentences(
        sentences, tfidf["matrix"].sum(axis=0), experiment_sentence_count
    )
    predictions = {
        "Lead-3": " ".join(sentences[:experiment_sentence_count]),
        "TF-IDF": tfidf_summary,
    }
    for k in experiment_k_values:
        scores = semantic_scores(singular_values, vt, k)
        selected_indices, prediction = select_sentences(sentences, scores, experiment_sentence_count)
        predictions[f"LSI (k={k})"] = prediction

    for method, prediction in predictions.items():
        metrics = rouge_scores(record["highlights"], prediction)
        evaluation_rows.append({"id": record["id"], "Phương pháp": method, **metrics})
        summary_rows.append({"id": record["id"], "Phương pháp": method,
                             "Tham chiếu": record["highlights"], "Tóm tắt": prediction})

evaluation = pd.DataFrame(evaluation_rows)
generated_summaries = pd.DataFrame(summary_rows)
average_scores = evaluation.groupby("Phương pháp", sort=False)[["ROUGE-1", "ROUGE-2", "ROUGE-L"]].mean()

### Trung bình ROUGE F1 trên toàn bộ tập test

In [63]:
average_scores

,ROUGE-1,ROUGE-2,ROUGE-L
Phương pháp,,,
Lead-3,0.288362,0.123862,0.213154
TF-IDF,0.212593,0.065363,0.143197
LSI (k=1),0.291026,0.127467,0.212445
LSI (k=2),0.222393,0.061687,0.154603
LSI (k=3),0.239495,0.076132,0.164873


## 3.4. Ví dụ

In [64]:
example_index = 0
example_id = records[example_index]["id"]

### Bài báo

In [65]:
print(records[example_index]["article"])

(CNN)The Palestinian Authority officially became the 123rd member of the International Criminal Court on Wednesday, a step that gives the court jurisdiction over alleged crimes in Palestinian territories. The formal accession was marked with a ceremony at The Hague, in the Netherlands, where the court is based. The Palestinians signed the ICC's founding Rome Statute in January, when they also accepted its jurisdiction over alleged crimes committed "in the occupied Palestinian territory, including East Jerusalem, since June 13, 2014." Later that month, the ICC opened a preliminary examination into the situation in Palestinian territories, paving the way for possible war crimes investigations against Israelis. As members of the court, Palestinians may be subject to counter-charges as well. Israel and the United States, neither of which is an ICC member, opposed the Palestinians' efforts to join the body. But Palestinian Foreign Minister Riad al-Malki, speaking at Wednesday's ceremony, sa

### Tóm tắt tham chiếu

In [66]:
print(records[example_index]["highlights"])

Membership gives the ICC jurisdiction over alleged crimes committed in Palestinian territories since last June .
Israel and the United States opposed the move, which could open the door to war crimes investigations against Israelis .


### Tóm tắt của từng phương pháp

In [67]:
example_summaries = generated_summaries.loc[generated_summaries["id"].eq(example_id)]
example_summaries[["Phương pháp", "Tóm tắt"]]

,Phương pháp,Tóm tắt
0,Lead-3,(CNN)The Palestinian Authority officially beca...
1,TF-IDF,The Palestinians signed the ICC's founding Rom...
2,LSI (k=1),(CNN)The Palestinian Authority officially beca...
3,LSI (k=2),"As members of the court, Palestinians may be s..."
4,LSI (k=3),"As members of the court, Palestinians may be s..."


### ROUGE của ví dụ

In [68]:
evaluation.loc[evaluation["id"].eq(example_id)].drop(columns="id")

,Phương pháp,ROUGE-1,ROUGE-2,ROUGE-L
0,Lead-3,0.295652,0.159292,0.226087
1,TF-IDF,0.236111,0.084507,0.208333
2,LSI (k=1),0.295652,0.159292,0.226087
3,LSI (k=2),0.197531,0.050633,0.148148
4,LSI (k=3),0.177778,0.022727,0.155556


## 3.5. Nhận xét kết quả

In [69]:
best_method = average_scores["ROUGE-L"].idxmax()
lsi_scores = average_scores.loc[[f"LSI (k={k})" for k in experiment_k_values]]
best_lsi = lsi_scores["ROUGE-L"].idxmax()
lead_difference = average_scores.loc[best_lsi, "ROUGE-L"] - average_scores.loc["Lead-3", "ROUGE-L"]

print("Phương pháp đạt ROUGE-L cao nhất trên tập mẫu:", best_method)
print("Cấu hình LSI đạt ROUGE-L cao nhất trên tập mẫu:", best_lsi)
print(f"Chênh lệch ROUGE-L của cấu hình LSI đó so với Lead-3: {lead_difference:+.4f}")

Phương pháp đạt ROUGE-L cao nhất trên tập mẫu: Lead-3
Cấu hình LSI đạt ROUGE-L cao nhất trên tập mẫu: LSI (k=1)
Chênh lệch ROUGE-L của cấu hình LSI đó so với Lead-3: -0.0007
